In [3]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import plotly.express as px
from flipside import Flipside
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from memory_profiler import profile
import json
import csv
import os
import logging
import sys
from importlib import reload
from pymongo import MongoClient, UpdateOne
from web3 import Web3
import psycopg2
from psycopg2.extras import execute_values
from io import StringIO
from typing import Any, Dict
from keys import KEYS

In [4]:
# logging configurations
reload(logging)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

## Extract Raw Data

### Data Extraction from Flipside Crypto

In [5]:
# Initilize Flipside Client
flipside_key = KEYS['flipside_key']
flipside = Flipside(flipside_key, "https://api-v2.flipsidecrypto.xyz")

In [6]:
def format_query(query_path: str, params: dict) -> str:
    try:
        with open(query_path, "r") as file:
            query = file.read()
        formatted_query = query.format(**params)
        return formatted_query
    except Exception as e:
        raise ValueError(f"format query error: {e}")

In [7]:
def createQueryRun(query : str, api_key:str = flipside_key) -> str :
    
    
    url = "https://api-v2.flipsidecrypto.xyz/json-rpc"

    # Request headers
    headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key
        }

    # Request payload
    payload = {
        "jsonrpc": "2.0",
        "method": "createQueryRun",
        "params": [
            {
                "resultTTLHours": 1,
                "maxAgeMinutes": 0,
                "sql": query ,
                "tags": {
                    "source": "postman-demo",
                    "env": "test"
                },
                "dataSource": "snowflake-default",
                "dataProvider": "flipside"
            }
        ],
        "id": 1
    }

    # Submit createQueryRun request
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        if response.status_code == 200:
            logging.info("Query run created successfully!")
            logging.debug(response.json())  # Output the response
            return  response.json()['result']['queryRequest']['queryRunId']
        else:
            raise requests.exceptions.HTTPError(
                f"Unexpected status code: {response.status_code}. Details: {response.text}" )
    except Exception as e:
        logging.error(f" createQueryRun Error: {e}")
    

In [8]:
def getQueryRun(queryRunId:str , api_key:str = flipside_key) -> str:
    
    url = "https://api-v2.flipsidecrypto.xyz/json-rpc"

    # Request headers
    headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key
        }
    
    payload = {
    "jsonrpc": "2.0",
    "method": "getQueryRun",
    "params": [
        {
            "queryRunId": queryRunId
        }
    ],
    "id": 1
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        logging.debug(f'getQueryRun state {response.json()['result']['queryRun']['state']}')
        return response.json()['result']['queryRun']['state']
        
    except Exception as e:
        logging.error(f" getQueryRun Error: {e}")

In [9]:
def queryresult_Pagination(queryRunId:str, page_size:int = 70000) -> list:
       
    current_page_number = 1
    total_pages = 3

    all_rows = []

    while current_page_number <= total_pages:

        try:
            results = flipside.get_query_results(
                queryRunId,
                page_number=current_page_number,
                page_size=page_size         
            )
       
            if results.records:
                total_pages = results.page.totalPages
                all_rows.extend(results.records)
                logging.debug(f"Current page number: {current_page_number} Total Pages: {total_pages}, Rows Retrieved: {len(results.records)}")
            else: 
                logging.warning('No record')
                break

        except Exception as e:
            logging.error(f" Pagination Error: {e}")
            return None
        
        current_page_number += 1

    logging.info(f"Total Pages: {total_pages}, Rows Retrieved: {len(all_rows)}")
        
        
    return all_rows

In [10]:
def extract_flipsidecrypto_data(query_path:str, params: dict , api_key = flipside_key, retry_time:int = 90 ,timeout:int = 600 ) -> list:
    
    try:
        logging.info(f'Start query with params:{params}')
        query = format_query(query_path,params)
        queryRunId = createQueryRun(query,api_key)

        state = None
        start_time = time.time()

        while state != 'QUERY_STATE_SUCCESS':
            
            state = getQueryRun(queryRunId,api_key)

            if state == 'QUERY_STATE_SUCCESS':
                 break 

            elif state in ['QUERY_STATE_FAILED', 'QUERY_STATE_CANCELED']:
                raise RuntimeError(f"Query execution failed or was canceled. State: {state}")
            
            elif state in ['QUERY_STATE_STREAMING_RESULTS', 'QUERY_STATE_RUNNING', 'QUERY_STATE_READY']:
                if time.time() - start_time > timeout:
                    raise TimeoutError("Query execution exceeded timeout limit.")
                
                logging.info(f"Wainting query excution")
                logging.debug(f"retry after {retry_time} sec")

                time.sleep(retry_time)

            else: raise ValueError(f"Unexpected query state: {state}")

            
        result = queryresult_Pagination(queryRunId)

    except TimeoutError as e:
        logging.error(f"Timeout Error: {e}")
        return None
    except RuntimeError as e:
        logging.error(f"Runtime Error: {e}")
        return None
    except Exception as e:
        logging.error(f" state Error: {e}")
        return None
    
                   

    return result

In [11]:
def get_start_block_number(pool: str, file_path: str, default: int = 0) -> int:
    try:
        with open(file_path, 'r') as file:
            block_numbers = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        logging.error(f"Error reading or parsing file {file_path}: {e}")
        return default  
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
        return default  

    block_number = block_numbers.get(pool, default)
    if isinstance(block_number, int):
        return block_number
    else:
        logging.warning(f"Invalid block number for pool '{pool}': {block_number}. Returning default value: {default}")
        return default

In [12]:
def update_start_block_number(data:list, file_path:str) -> None :
    try:
        try:
            with open(file_path, 'r') as file:
                block_numbers = json.load(file)
        except (FileNotFoundError, json.JSONDecodeError):
            block_numbers = {}

        for pool, events in data.items():
            if events:

                last_block_number = max(event['block_number'] for event in events)

                block_numbers[pool] = last_block_number

                logging.debug(f"Block number for pool {pool} updated to {last_block_number}")
            else:
                logging.warning(f"No events found for pool {pool}, skipping.")

    except Exception as e:
        logging.error(f"Error updating block numbers: {e}")
        
    try:
        
        with open(file_path, 'w') as file:
                json.dump(block_numbers, file, indent=4)
                logging.info(f"block number config File updated")
                
    except Exception as e:
        logging.error(f"Error updating block number config File: {e}")
        
    return None

In [13]:
def fetch_positionsData(pool_address:str, query_path:str, block_number_config_file_path:str) -> list:
    try: 
        block_number = get_start_block_number(pool_address,block_number_config_file_path)
        logging.info(f"querying data for pool : {pool_address} starting from block number: {block_number}")
        position_data = extract_flipsidecrypto_data(query_path, params= {'pool_address': pool_address,'block_number':block_number} )
        logging.info(f"position data fetched successfully for pool {pool_address}, Rows Retrieved: {len(position_data)}")
    except Exception as e:
        logging.error(f"Error fetching position data for pool: {pool_address}: {e}")
    return position_data

In [14]:
def search_pools(query_path:str,numberofpools:int):    

    ### get Top {numberofpools} Pools with higest volume 1 month period

    pool_search_query_params = {'NumberOfpools':numberofpools}
    extracted_pools = extract_flipsidecrypto_data(query_path, pool_search_query_params)
    pools_list = [pool['pool_address'] for pool in extracted_pools]
    
    return pools_list

In [15]:
def fetch_positionData_all_pools(pool_addresses:list,query_path_positionData:str,file_path_block_number:str) -> list:
    all_results = {}
    try:   
        with ThreadPoolExecutor() as executor:
            future_to_pool = {executor.submit(fetch_positionsData, pool,query_path_positionData,file_path_block_number): pool for pool in pool_addresses}
            for future in as_completed(future_to_pool):
                pool = future_to_pool[future]
                result = future.result()
                all_results[pool] = result
    except Exception as e:
        logging.error(f"error in  fetching  positionData: {e}")

    update_start_block_number(all_results,file_path_block_number)
    flattened_values = [item for sublist in all_results.values() for item in sublist]           
    return flattened_values

In [16]:
def normalize_pool_addresses(pool_addresses_list:list) -> tuple:
    try:
        pool_addresses_tuple = tuple(pool_addresses_list)
        if len(pool_addresses_tuple) == 1:
            pool_addresses_tuple = (pool_addresses_tuple[0], pool_addresses_tuple[0])
        return pool_addresses_tuple
    except Exception as e:
        logging.error(f"Error Normalizing addresess list: {e}")
    

In [ ]:

pool_search_query_path = r'sql_queries\search_pools_by_volume_query.sql'
pools_list = search_pools(pool_search_query_path,1)


2024-10-19 14:22:58 - INFO - Start query with params:{'NumberOfpools': 1}
2024-10-19 14:22:59 - INFO - Query run created successfully!
2024-10-19 14:22:59 - INFO - Wainting query excution


In [ ]:
pools_list

['0x4548280ac92507c9092a511c7396cbea78fa9e49']

In [ ]:
positin_data_query_path = r'sql_queries\get_position_data_query.sql'
block_number_config_file_path = 'configs/block_number_config.json'
positions_extracted_data = fetch_positionData_all_pools(pools_list,positin_data_query_path,block_number_config_file_path)

2024-10-19 16:15:03 - INFO - querying data for pool : 0x4548280ac92507c9092a511c7396cbea78fa9e49 starting from block number: 20891324
2024-10-19 16:15:03 - INFO - Start query with params:{'pool_address': '0x4548280ac92507c9092a511c7396cbea78fa9e49', 'block_number': 20891324}
2024-10-19 16:15:04 - INFO - Query run created successfully!
2024-10-19 16:15:04 - INFO - Wainting query excution
2024-10-19 16:16:35 - WARNING - No record
2024-10-19 16:16:35 - INFO - Total Pages: 3, Rows Retrieved: 0
2024-10-19 16:16:35 - INFO - position data fetched successfully for pool 0x4548280ac92507c9092a511c7396cbea78fa9e49, Rows Retrieved: 0
2024-10-19 16:16:35 - WARNING - No events found for pool 0x4548280ac92507c9092a511c7396cbea78fa9e49, skipping.
2024-10-19 16:16:35 - INFO - block number config File updated


In [ ]:
positions_extracted_data

[]

In [ ]:
pool_info_query_path = r'sql_queries\get_pool_info_query.sql'
pools_list_tuple = normalize_pool_addresses(pools_list)
pool_info_data = extract_flipsidecrypto_data(pool_info_query_path, params= {'pool_address': pools_list_tuple} )

2024-10-19 16:16:35 - INFO - Start query with params:{'pool_address': ('0x4548280ac92507c9092a511c7396cbea78fa9e49', '0x4548280ac92507c9092a511c7396cbea78fa9e49')}
2024-10-19 16:16:36 - INFO - Query run created successfully!
2024-10-19 16:16:36 - INFO - Wainting query excution
2024-10-19 16:18:07 - INFO - Total Pages: 1, Rows Retrieved: 1


In [ ]:
pool_info_data

[{'pool_address': '0x4548280ac92507c9092a511c7396cbea78fa9e49',
  'token0': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',
  'token1': '0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf',
  'fee': 3000,
  'tickspacing': 60,
  '__row_index': 0}]

### Loading Raw data to MongoDB

In [69]:
def initialize_mongo_db(host:str, database_name:str):
    try:
        client = MongoClient(host)
        db = client[database_name]
    except Exception as e:
        logging.error(f"Error in initiating Mongo client: {e}")    

    return client,db

In [ ]:
def bulk_upsert_mongodb(host:str, database_name:str, collection_name:str, data:pd.DataFrame, unique_keys:list) -> Any:
    try:
        client, db = initialize_mongo_db(host, database_name)
        if client is None or db is None:
            logging.error("Failed to connect to MongoDB.")
            return None
        else: 
            logging.debug('Connected to mongoDB successfully')
            

        collection = db[collection_name]

        
        bulk_operations = [
            UpdateOne(
                {key: record[key] for key in unique_keys if key in record}, 
                {"$setOnInsert": record},
                upsert=True
            )
            for record in data
        ]

        result = None
        if bulk_operations:
            result = collection.bulk_write(bulk_operations, ordered=False)
            logging.debug(f"Inserted: {result.upserted_count}")
            if result.upserted_count > 0:
                logging.info("Mongodb: Data inserted successfully!")
            else: 
                logging.info("Mongodb: No new data to be inserted!")
        return result
    
            
    
    except Exception as e:
        logging.error(f"Error in bulk upsert operation: {e}")
        return None

    finally:
        if client:
            client.close()
            logging.debug("MongoDB: Client closed!") 


In [5]:
def load_config(config_path="configs/mongodb_config.json"):
    with open(config_path, "r") as file:
        return json.load(file)["database"]

In [6]:
config = load_config()
host = config["host"]
database_name = config["name"]
collections = config["collections"]
print(collections)

{'collection_postion_Data': {'collection_name': 'positionsdata_raw', 'unique_keys': ['block_number', 'tx_hash', 'event_index']}, 'collection_pool_info_Data': {'collection_name': 'poolinfo_data', 'unique_keys': ['pool_address']}}


In [7]:
collection_postion_Data = config["collections"]["collection_postion_Data"]["collection_name"]
collection_postion_Data_keys = config["collections"]["collection_postion_Data"]["unique_keys"]
collection_pool_info_Data = config["collections"]["collection_pool_info_Data"]["collection_name"]
collection_pool_info_Data_keys = config["collections"]["collection_pool_info_Data"]["unique_keys"]

In [ ]:
# insert position data
upsert_result_position_data = bulk_upsert_mongodb(host,database_name, collection_postion_Data, positions_extracted_data,collection_postion_Data_keys)

In [ ]:
# insert position data
upsert_result_poolinfo_data = bulk_upsert_mongodb(host,database_name, collection_pool_info_Data, pool_info_data,collection_pool_info_Data_keys)

2024-10-19 16:18:07 - INFO - Mongodb: Data inserted successfully!


## Transform Data

In [6]:
etherscan_key = KEYS['etherscan_key']

In [7]:
def load_config(config_path="configs/mongodb_config.json"):
    with open(config_path, "r") as file:
        return json.load(file)["database"]

In [8]:
config = load_config()
host = config["host"]
database_name = config["name"]
collections = config["collections"]
print(collections)

{'collection_postion_Data': {'collection_name': 'positionsdata_raw', 'unique_keys': ['block_number', 'tx_hash', 'event_index']}, 'collection_pool_info_Data': {'collection_name': 'poolinfo_data', 'unique_keys': ['pool_address']}}


In [9]:
collection_postion_Data = config["collections"]["collection_postion_Data"]["collection_name"]
collection_postion_Data_keys = config["collections"]["collection_postion_Data"]["unique_keys"]
collection_pool_info_Data = config["collections"]["collection_pool_info_Data"]["collection_name"]
collection_pool_info_Data_keys = config["collections"]["collection_pool_info_Data"]["unique_keys"]

In [10]:
def initialize_mongo_db(host:str, database_name:str):
    try:
        client = MongoClient(host)
        db = client[database_name]
    except Exception as e:
        logging.error(f"Error in initiating Mongo client: {e}")    

    return client,db

In [11]:
def load_mongo_collection(host:str, database_name:str, collection_name:str) -> pd.DataFrame:
    try:
        client, db = initialize_mongo_db(host, database_name)
        if client is None or db is None:
            logging.error("Failed to connect to MongoDB.")
            return None
        else: 
            logging.debug('Connected to mongoDB successfully')
            

        collection = db[collection_name]

        collection_data_df = pd.DataFrame(list(collection.find()))

        return collection_data_df

    except Exception as e:
        logging.error(f"Error in loading data from mongo: {e}")
        return None

In [12]:
def get_ABI(contract_address:str, api_key:str = etherscan_key) -> dict[str, any]:
    url = "https://api.etherscan.io/v2/api"
    params = {
        "chainid": 1,
        "module": "contract",
        "action": "getabi",
        "address": contract_address,
        "apikey": api_key
    }

    try:
        response = requests.get(url, params=params)
        return json.loads(response.json()['result'])
    except Exception as e:
        logging.error(f" request Error: {e}")

In [13]:
from web3.contract import Contract

In [14]:
def create_contract_instance(contract_address:str)->Contract:
    try:
        web3 = Web3()
        ABI = get_ABI(contract_address)
        logging.debug(f"contract ABI: {ABI}")
        contract_instance = web3.eth.contract(abi=ABI)
        logging.debug(f"contract instance: {contract_instance}")
        logging.info("contract instance created successfully")
        return contract_instance
    except Exception as e:
        logging.error(f" request Error: {e}")

In [15]:
def aggregate_logs(df:pd.DataFrame, data:str, topics:str, event_index:int, tx_index:int, tx_hash:str, contract_address:str, block_hash:str, block_number:int) -> dict:
    try:
        log = {
        'data': df[str(data)],
        'topics': df[str(topics)],
        'logIndex': df[event_index],
        'transactionIndex': df[tx_index],
        'transactionHash': df[tx_hash],
        'address': df[str(contract_address)],
        'blockHash': df[str(block_hash)],
        'blockNumber': df[block_number]
                }
        logging.debug('logs aggregated sucessfully!')
        return log
    except Exception as e:
        logging.error(f"Error in log aggregation: {e}")

In [16]:
def decode_event(df:pd.DataFrame, contract_instance:Contract, event_column:str, log_column:str) ->dict:
    try:
        event = getattr(contract_instance.events, df[event_column])()
        processed_log = event.process_log(df[str(log_column)])
        logging.debug('log processed sucessfully!')
        return processed_log
    except AttributeError:
        logging.error(f"Error: Event {df[str(event_column)]} does not exist.")
    except Exception as e:
        logging.error(f"Error processing log for event {df[str(event_column)]}: {e}")

In [17]:
positions_data_raw = load_mongo_collection(host,database_name, collection_postion_Data)
positions_data_raw.info()
positions_data_raw.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   _id                  14 non-null     object
 1   block_number         14 non-null     int64 
 2   event_index          14 non-null     int64 
 3   tx_hash              14 non-null     object
 4   __row_index          14 non-null     int64 
 5   block_hash           14 non-null     object
 6   block_timestamp      14 non-null     object
 7   contract_address     14 non-null     object
 8   data                 14 non-null     object
 9   event_name           14 non-null     object
 10  origin_from_address  14 non-null     object
 11  origin_to_address    14 non-null     object
 12  pool_address         14 non-null     object
 13  topic0               14 non-null     object
 14  topics               14 non-null     object
 15  tx_index             14 non-null     int64 
dtypes: int64(4

,_id,block_number,event_index,tx_hash,__row_index,block_hash,block_timestamp,contract_address,data,event_name,origin_from_address,origin_to_address,pool_address,topic0,topics,tx_index
0,67bf53fa36cbc624db1d918f,20733495,419,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,0,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,2024-09-12T08:59:23.000Z,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x00000000000000000000000000000000000000000000...,DecreaseLiquidity,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x26f6a048ee9138f2c0ce266f322cb99228e8d619ae2b...,[0x26f6a048ee9138f2c0ce266f322cb99228e8d619ae2...,146
1,67bf53fa36cbc624db1d9190,20733432,196,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,1,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x00000000000000000000000000000000000000000000...,Burn,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acf...,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,121
2,67bf53fa36cbc624db1d9191,20733432,198,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,2,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x00000000000000000000000000000000000000000000...,Burn,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acf...,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,121
3,67bf53fa36cbc624db1d9192,20733495,418,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,3,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,2024-09-12T08:59:23.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x00000000000000000000000000000000000000000000...,Burn,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acf...,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,146
4,67bf53fa36cbc624db1d9193,20722319,96,0x44f9bb7c1cc5057a0888329e5399cef339182c2d6806...,4,0x5643e2956fa116379a793bc3156e9980a932c309e65b...,2024-09-10T19:32:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x000000000000000000000000c36442b4a4522e871399...,Mint,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85...,[0x7a53080ba414158be7ec69b987b5fb7d07dee101fe8...,7


In [18]:
pool_info_data_raw = load_mongo_collection(host,database_name, collection_pool_info_Data)
pool_info_data_raw.info()
pool_info_data_raw.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   _id           8 non-null      object
 1   pool_address  8 non-null      object
 2   __row_index   8 non-null      int64 
 3   fee           8 non-null      int64 
 4   tickspacing   8 non-null      int64 
 5   token0        8 non-null      object
 6   token1        8 non-null      object
dtypes: int64(3), object(4)
memory usage: 580.0+ bytes


,_id,pool_address,__row_index,fee,tickspacing,token0,token1
0,67bb539f36cbc624db1d70bc,0x914565e885cb9df773e83b0ecbdabb36c0de3b10,0,10000,200,0x5ee84583f67d5ecea5420dbb42b462896e7f8d06,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2
1,67bb539f36cbc624db1d70bd,0x50fe1432a9127b25d81ba12d739b744f84111134,1,10000,200,0x808507121b80c02388fad14726482e061b8da827,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2
2,67bba03b36cbc624db1d7365,0x30b8f02dc5c0e049dfa5b570ecc83462698f64aa,0,100,1,0x7122985656e38bdc0302db86685bb972b145bd3c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2
3,67bba03b36cbc624db1d7366,0x85e23ea64ea953f4a952ec259afa8d584a9d3770,1,3000,60,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xe973e453977195422b48e1852a207b7ee9c913c7
4,67bbae3c36cbc624db1d918d,0xe15e6583425700993bd08f51bf6e7b73cd5da91b,0,3000,60,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,0x3472a5a71965499acd81997a54bba8d852c6e53d


In [19]:
UniswapV3Pool_contract_instance = create_contract_instance(contract_address="0xe8f7c89c5efa061e340f2d2f206ec78fd8f7e124")
NonfungiblePositionManager_contract_instance = create_contract_instance(contract_address="0xc36442b4a4522e871399cd717abdd847ab11fe88")

2024-10-19 17:11:56 - INFO - contract instance created successfully
2024-10-19 17:11:56 - INFO - contract instance created successfully


In [20]:
positions_data_raw['log'] = positions_data_raw.apply(lambda row: aggregate_logs(
    row,
    data='data',
    topics='topics',
    event_index='event_index',
    tx_index='tx_index',
    tx_hash='tx_hash',
    contract_address='contract_address',
    block_hash='block_hash',
    block_number='block_number'
), axis=1)

In [21]:
def decode_data(data:pd.DataFrame, pool_topic0:list, nft_topic0:list) -> pd.DataFrame:
    try:

        data.loc[data['topic0'].isin(pool_topic0), "Processed_Log"] = data.loc[
            data['topic0'].isin(pool_topic0)
        ].apply(lambda row: decode_event(row, UniswapV3Pool_contract_instance, 'event_name', 'log'), axis=1)
        data.loc[data['topic0'].isin(nft_topic0), "Processed_Log"] = data.loc[
            data['topic0'].isin(nft_topic0)
        ].apply(lambda row: decode_event(row, NonfungiblePositionManager_contract_instance, 'event_name', 'log'), axis=1)
        data['args'] = data['Processed_Log'].apply(lambda x: x.get('args'))

        logging.info("Data Decoded sucessfully!")
        return data
    except Exception as e:
        logging.error(f" Data decoding Error: {e}")

    


In [22]:
pool_topic0 = ['0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85488f0853ae16239d0bde','0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acfd67e028cd568da98982c']
nft_topic0 = ['0x3067048beee31b25b2f1681f88dac838c8bba36af25bfb2b7cf7473a5847e35f','0x26f6a048ee9138f2c0ce266f322cb99228e8d619ae2bff30c67f8dcf9d2377b4']

In [23]:
positions_data_decoded = decode_data(positions_data_raw, pool_topic0, nft_topic0)
positions_data_decoded.info()
positions_data_decoded.head()

2024-10-19 17:12:03 - INFO - Data Decoded sucessfully!


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   _id                  14 non-null     object
 1   block_number         14 non-null     int64 
 2   event_index          14 non-null     int64 
 3   tx_hash              14 non-null     object
 4   __row_index          14 non-null     int64 
 5   block_hash           14 non-null     object
 6   block_timestamp      14 non-null     object
 7   contract_address     14 non-null     object
 8   data                 14 non-null     object
 9   event_name           14 non-null     object
 10  origin_from_address  14 non-null     object
 11  origin_to_address    14 non-null     object
 12  pool_address         14 non-null     object
 13  topic0               14 non-null     object
 14  topics               14 non-null     object
 15  tx_index             14 non-null     int64 
 16  log       

,_id,block_number,event_index,tx_hash,__row_index,block_hash,block_timestamp,contract_address,data,event_name,origin_from_address,origin_to_address,pool_address,topic0,topics,tx_index,log,Processed_Log,args
0,67bf53fa36cbc624db1d918f,20733495,419,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,0,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,2024-09-12T08:59:23.000Z,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x00000000000000000000000000000000000000000000...,DecreaseLiquidity,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x26f6a048ee9138f2c0ce266f322cb99228e8d619ae2b...,[0x26f6a048ee9138f2c0ce266f322cb99228e8d619ae2...,146,{'data': '0x0000000000000000000000000000000000...,"{'args': {'tokenId': 807811, 'liquidity': 5040...","{'tokenId': 807811, 'liquidity': 5040283203992..."
1,67bf53fa36cbc624db1d9190,20733432,196,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,1,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x00000000000000000000000000000000000000000000...,Burn,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acf...,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,121,{'data': '0x0000000000000000000000000000000000...,{'args': {'owner': '0xC36442b4a4522E871399CD71...,{'owner': '0xC36442b4a4522E871399CD717aBDD847A...
2,67bf53fa36cbc624db1d9191,20733432,198,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,2,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x00000000000000000000000000000000000000000000...,Burn,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acf...,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,121,{'data': '0x0000000000000000000000000000000000...,{'args': {'owner': '0xC36442b4a4522E871399CD71...,{'owner': '0xC36442b4a4522E871399CD717aBDD847A...
3,67bf53fa36cbc624db1d9192,20733495,418,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,3,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,2024-09-12T08:59:23.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x00000000000000000000000000000000000000000000...,Burn,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acf...,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,146,{'data': '0x0000000000000000000000000000000000...,{'args': {'owner': '0xC36442b4a4522E871399CD71...,{'owner': '0xC36442b4a4522E871399CD717aBDD847A...
4,67bf53fa36cbc624db1d9193,20722319,96,0x44f9bb7c1cc5057a0888329e5399cef339182c2d6806...,4,0x5643e2956fa116379a793bc3156e9980a932c309e65b...,2024-09-10T19:32:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x000000000000000000000000c36442b4a4522e871399...,Mint,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85...,[0x7a53080ba414158be7ec69b987b5fb7d07dee101fe8...,7,{'data': '0x000000000000000000000000c36442b4a4...,{'args': {'owner': '0xC36442b4a4522E871399CD71...,{'owner': '0xC36442b4a4522E871399CD717aBDD847A...


In [24]:
positions_data_decoded

,_id,block_number,event_index,tx_hash,__row_index,block_hash,block_timestamp,contract_address,data,event_name,origin_from_address,origin_to_address,pool_address,topic0,topics,tx_index,log,Processed_Log,args
0,67bf53fa36cbc624db1d918f,20733495,419,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,0,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,2024-09-12T08:59:23.000Z,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x00000000000000000000000000000000000000000000...,DecreaseLiquidity,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x26f6a048ee9138f2c0ce266f322cb99228e8d619ae2b...,[0x26f6a048ee9138f2c0ce266f322cb99228e8d619ae2...,146,{'data': '0x0000000000000000000000000000000000...,"{'args': {'tokenId': 807811, 'liquidity': 5040...","{'tokenId': 807811, 'liquidity': 5040283203992..."
1,67bf53fa36cbc624db1d9190,20733432,196,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,1,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x00000000000000000000000000000000000000000000...,Burn,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acf...,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,121,{'data': '0x0000000000000000000000000000000000...,{'args': {'owner': '0xC36442b4a4522E871399CD71...,{'owner': '0xC36442b4a4522E871399CD717aBDD847A...
2,67bf53fa36cbc624db1d9191,20733432,198,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,2,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x00000000000000000000000000000000000000000000...,Burn,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acf...,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,121,{'data': '0x0000000000000000000000000000000000...,{'args': {'owner': '0xC36442b4a4522E871399CD71...,{'owner': '0xC36442b4a4522E871399CD717aBDD847A...
3,67bf53fa36cbc624db1d9192,20733495,418,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,3,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,2024-09-12T08:59:23.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x00000000000000000000000000000000000000000000...,Burn,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acf...,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,146,{'data': '0x0000000000000000000000000000000000...,{'args': {'owner': '0xC36442b4a4522E871399CD71...,{'owner': '0xC36442b4a4522E871399CD717aBDD847A...
4,67bf53fa36cbc624db1d9193,20722319,96,0x44f9bb7c1cc5057a0888329e5399cef339182c2d6806...,4,0x5643e2956fa116379a793bc3156e9980a932c309e65b...,2024-09-10T19:32:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x000000000000000000000000c36442b4a4522e871399...,Mint,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85...,[0x7a53080ba414158be7ec69b987b5fb7d07dee101fe8...,7,{'data': '0x000000000000000000000000c36442b4a4...,{'args': {'owner': '0xC36442b4a4522E871399CD71...,{'owner': '0xC36442b4a4522E871399CD717aBDD847A...
5,67bf53fa36cbc624db1d9194,20831017,253,0x38651d8b15f6b54645e486debf3a2a4908137d1bfe5b...,5,0x161ed2d72668674a98cdd77da68842ecb83af2f6e53b...,2024-09-25T23:51:23.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x00000000000000000000000000000000000000000000...,Burn,0x366aa56191e89d219ac36b33406fce85da1e7554,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acf...,[0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908ac...,94,{'data': '0x000000000000000000000000000000

In [25]:
pool_data = positions_data_decoded[positions_data_decoded['topic0'].isin(pool_topic0)][['block_timestamp',"pool_address",'Processed_Log']].reset_index(drop=True)
pool_data.info()
pool_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   block_timestamp  8 non-null      object
 1   pool_address     8 non-null      object
 2   Processed_Log    8 non-null      object
dtypes: object(3)
memory usage: 324.0+ bytes


,block_timestamp,pool_address,Processed_Log
0,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,{'args': {'owner': '0xC36442b4a4522E871399CD71...
1,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,{'args': {'owner': '0xC36442b4a4522E871399CD71...
2,2024-09-12T08:59:23.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,{'args': {'owner': '0xC36442b4a4522E871399CD71...
3,2024-09-10T19:32:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,{'args': {'owner': '0xC36442b4a4522E871399CD71...
4,2024-09-25T23:51:23.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,{'args': {'owner': '0xC36442b4a4522E871399CD71...


In [26]:
normalized_processed_log_pool = pd.json_normalize(pool_data['Processed_Log'])
normalized_processed_log_pool.columns = normalized_processed_log_pool.columns.str.replace(r'^args\.', '', regex=True)
pool_normalized_data = pd.concat([pool_data[['block_timestamp',"pool_address"]], normalized_processed_log_pool], axis=1)
pool_normalized_data.info()
pool_normalized_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   block_timestamp   8 non-null      object
 1   pool_address      8 non-null      object
 2   event             8 non-null      object
 3   logIndex          8 non-null      int64 
 4   transactionIndex  8 non-null      int64 
 5   transactionHash   8 non-null      object
 6   address           8 non-null      object
 7   blockHash         8 non-null      object
 8   blockNumber       8 non-null      int64 
 9   owner             8 non-null      object
 10  tickLower         8 non-null      int64 
 11  tickUpper         8 non-null      int64 
 12  amount            8 non-null      int64 
 13  amount0           8 non-null      int64 
 14  amount1           8 non-null      int64 
 15  sender            2 non-null      object
dtypes: int64(8), object(8)
memory usage: 1.1+ KB


,block_timestamp,pool_address,event,logIndex,transactionIndex,transactionHash,address,blockHash,blockNumber,owner,tickLower,tickUpper,amount,amount0,amount1,sender
0,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,Burn,196,121,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,20733432,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,193380,200280,434507173001454,689653790,2498368860914347040,NaN
1,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,Burn,198,121,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,20733432,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,193380,200280,0,0,0,NaN
2,2024-09-12T08:59:23.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,Burn,418,146,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,20733495,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,193380,200340,50402832039922,86762159,289810787703649019,NaN
3,2024-09-10T19:32:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,Mint,96,7,0x44f9bb7c1cc5057a0888329e5399cef339182c2d6806...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x5643e2956fa116379a793bc3156e9980a932c309e65b...,20722319,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,193380,200340,100805664079844,172710488,579999999590987635,0xC36442b4a4522E871399CD717aBDD847Ab11FE88
4,2024-09-25T23:51:23.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,Burn,253,94,0x38651d8b15f6b54645e486debf3a2a4908137d1bfe5b...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0x161ed2d72668674a98cdd77da68842ecb83af2f6e53b...,20831017,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,193380,200340,50402832039922,209313180,235673801212928920,NaN


In [27]:
pool_curated_data = pool_normalized_data[['block_timestamp','blockHash','transactionHash','blockNumber','transactionIndex','logIndex','pool_address','owner','sender','amount','amount0','amount1','event','tickLower','tickUpper']].rename(columns={"event": "pool_event",'amount':"liquidity"})
pool_curated_data

,block_timestamp,blockHash,transactionHash,blockNumber,transactionIndex,logIndex,pool_address,owner,sender,liquidity,amount0,amount1,pool_event,tickLower,tickUpper
0,2024-09-12T08:46:47.000Z,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,20733432,121,196,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,434507173001454,689653790,2498368860914347040,Burn,193380,200280
1,2024-09-12T08:46:47.000Z,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,20733432,121,198,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,0,0,0,Burn,193380,200280
2,2024-09-12T08:59:23.000Z,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,20733495,146,418,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,50402832039922,86762159,289810787703649019,Burn,193380,200340
3,2024-09-10T19:32:47.000Z,0x5643e2956fa116379a793bc3156e9980a932c309e65b...,0x44f9bb7c1cc5057a0888329e5399cef339182c2d6806...,20722319,7,96,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,100805664079844,172710488,579999999590987635,Mint,193380,200340
4,2024-09-25T23:51:23.000Z,0x161ed2d72668674a98cdd77da68842ecb83af2f6e53b...,0x38651d8b15f6b54645e486debf3a2a4908137d1bfe5b...,20831017,94,253,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,50402832039922,209313180,235673801212928920,Burn,193380,200340
5,2024-09-24T10:57:59.000Z,0x9ff02be3eb793f000e68148ba750e51485a9b058e68d...,0xbcce8ffd844ea8a0cf240fb373ae03a753359ac9621e...,20820005,21,186,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,434507173001455,1798992173,2009536441754882992,Burn,193380,200280
6,2024-09-12T08:59:23.000Z,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,20733495,146,420,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,0,0,0,Burn,193380,200340
7,2024-09-10T19:34:35.000Z,0xb3aadb49c0e75a3e5481effafeaddd06a44244a51945...,0x714665f29dab6c383e5ecf0947a129d7d15726d85320...,20722328,119,162,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,869014346002909,1372291791,4999999999276128040,Mint,193380,200280


In [28]:
nft_data = positions_data_decoded[positions_data_decoded['topic0'].isin(nft_topic0)][['block_timestamp',"pool_address",'Processed_Log']].reset_index(drop=True)
nft_data.info()
nft_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   block_timestamp  6 non-null      object
 1   pool_address     6 non-null      object
 2   Processed_Log    6 non-null      object
dtypes: object(3)
memory usage: 276.0+ bytes


,block_timestamp,pool_address,Processed_Log
0,2024-09-12T08:59:23.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,"{'args': {'tokenId': 807811, 'liquidity': 5040..."
1,2024-09-24T10:57:59.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,"{'args': {'tokenId': 807812, 'liquidity': 4345..."
2,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,"{'args': {'tokenId': 807812, 'liquidity': 4345..."
3,2024-09-10T19:34:35.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,"{'args': {'tokenId': 807812, 'liquidity': 8690..."
4,2024-09-10T19:32:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,"{'args': {'tokenId': 807811, 'liquidity': 1008..."


In [29]:
normalized_processed_log_nft = pd.json_normalize(nft_data['Processed_Log'])
normalized_processed_log_nft.columns = normalized_processed_log_nft.columns.str.replace(r'^args\.', '', regex=True)
nft_normalized_data = pd.concat([nft_data[['block_timestamp',"pool_address"]], normalized_processed_log_nft], axis=1)
nft_normalized_data.info()
nft_normalized_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   block_timestamp   6 non-null      object
 1   pool_address      6 non-null      object
 2   event             6 non-null      object
 3   logIndex          6 non-null      int64 
 4   transactionIndex  6 non-null      int64 
 5   transactionHash   6 non-null      object
 6   address           6 non-null      object
 7   blockHash         6 non-null      object
 8   blockNumber       6 non-null      int64 
 9   tokenId           6 non-null      int64 
 10  liquidity         6 non-null      int64 
 11  amount0           6 non-null      int64 
 12  amount1           6 non-null      int64 
dtypes: int64(7), object(6)
memory usage: 756.0+ bytes


,block_timestamp,pool_address,event,logIndex,transactionIndex,transactionHash,address,blockHash,blockNumber,tokenId,liquidity,amount0,amount1
0,2024-09-12T08:59:23.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,DecreaseLiquidity,419,146,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,20733495,807811,50402832039922,86762159,289810787703649019
1,2024-09-24T10:57:59.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,DecreaseLiquidity,187,21,0xbcce8ffd844ea8a0cf240fb373ae03a753359ac9621e...,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x9ff02be3eb793f000e68148ba750e51485a9b058e68d...,20820005,807812,434507173001455,1798992173,2009536441754882992
2,2024-09-12T08:46:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,DecreaseLiquidity,197,121,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,20733432,807812,434507173001454,689653790,2498368860914347040
3,2024-09-10T19:34:35.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,IncreaseLiquidity,164,119,0x714665f29dab6c383e5ecf0947a129d7d15726d85320...,0xc36442b4a4522e871399cd717abdd847ab11fe88,0xb3aadb49c0e75a3e5481effafeaddd06a44244a51945...,20722328,807812,869014346002909,1372291791,4999999999276128040
4,2024-09-10T19:32:47.000Z,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,IncreaseLiquidity,98,7,0x44f9bb7c1cc5057a0888329e5399cef339182c2d6806...,0xc36442b4a4522e871399cd717abdd847ab11fe88,0x5643e2956fa116379a793bc3156e9980a932c309e65b...,20722319,807811,100805664079844,172710488,579999999590987635


In [30]:
nft_curated_data = nft_normalized_data[['blockHash','blockNumber','transactionHash','pool_address','liquidity','amount0','amount1','event','tokenId']].rename(columns={"event": "NFT_event"})
nft_curated_data

,blockHash,blockNumber,transactionHash,pool_address,liquidity,amount0,amount1,NFT_event,tokenId
0,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,20733495,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,50402832039922,86762159,289810787703649019,DecreaseLiquidity,807811
1,0x9ff02be3eb793f000e68148ba750e51485a9b058e68d...,20820005,0xbcce8ffd844ea8a0cf240fb373ae03a753359ac9621e...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,434507173001455,1798992173,2009536441754882992,DecreaseLiquidity,807812
2,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,20733432,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,434507173001454,689653790,2498368860914347040,DecreaseLiquidity,807812
3,0xb3aadb49c0e75a3e5481effafeaddd06a44244a51945...,20722328,0x714665f29dab6c383e5ecf0947a129d7d15726d85320...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,869014346002909,1372291791,4999999999276128040,IncreaseLiquidity,807812
4,0x5643e2956fa116379a793bc3156e9980a932c309e65b...,20722319,0x44f9bb7c1cc5057a0888329e5399cef339182c2d6806...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,100805664079844,172710488,579999999590987635,IncreaseLiquidity,807811
5,0x161ed2d72668674a98cdd77da68842ecb83af2f6e53b...,20831017,0x38651d8b15f6b54645e486debf3a2a4908137d1bfe5b...,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,50402832039922,209313180,235673801212928920,DecreaseLiquidity,807811


In [31]:
position_curated_data_prefinal = pool_curated_data.merge(nft_curated_data,how='left',left_on=['blockHash','blockNumber','transactionHash','pool_address','liquidity','amount0','amount1'],right_on=['blockHash','blockNumber','transactionHash','pool_address','liquidity','amount0','amount1'])
position_curated_data_prefinal.info()
position_curated_data_prefinal.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   block_timestamp   8 non-null      object 
 1   blockHash         8 non-null      object 
 2   transactionHash   8 non-null      object 
 3   blockNumber       8 non-null      int64  
 4   transactionIndex  8 non-null      int64  
 5   logIndex          8 non-null      int64  
 6   pool_address      8 non-null      object 
 7   owner             8 non-null      object 
 8   sender            2 non-null      object 
 9   liquidity         8 non-null      int64  
 10  amount0           8 non-null      int64  
 11  amount1           8 non-null      int64  
 12  pool_event        8 non-null      object 
 13  tickLower         8 non-null      int64  
 14  tickUpper         8 non-null      int64  
 15  NFT_event         6 non-null      object 
 16  tokenId           6 non-null      float64
dtypes

,block_timestamp,blockHash,transactionHash,blockNumber,transactionIndex,logIndex,pool_address,owner,sender,liquidity,amount0,amount1,pool_event,tickLower,tickUpper,NFT_event,tokenId
0,2024-09-12T08:46:47.000Z,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,20733432,121,196,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,434507173001454,689653790,2498368860914347040,Burn,193380,200280,DecreaseLiquidity,807812.0
1,2024-09-12T08:46:47.000Z,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,20733432,121,198,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,0,0,0,Burn,193380,200280,NaN,NaN
2,2024-09-12T08:59:23.000Z,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,20733495,146,418,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,50402832039922,86762159,289810787703649019,Burn,193380,200340,DecreaseLiquidity,807811.0
3,2024-09-10T19:32:47.000Z,0x5643e2956fa116379a793bc3156e9980a932c309e65b...,0x44f9bb7c1cc5057a0888329e5399cef339182c2d6806...,20722319,7,96,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,100805664079844,172710488,579999999590987635,Mint,193380,200340,IncreaseLiquidity,807811.0
4,2024-09-25T23:51:23.000Z,0x161ed2d72668674a98cdd77da68842ecb83af2f6e53b...,0x38651d8b15f6b54645e486debf3a2a4908137d1bfe5b...,20831017,94,253,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,50402832039922,209313180,235673801212928920,Burn,193380,200340,DecreaseLiquidity,807811.0


In [32]:
pool_info_curated_data = pool_info_data_raw[['pool_address','token0','token1','fee','tickspacing']]
pool_info_curated_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   pool_address  8 non-null      object
 1   token0        8 non-null      object
 2   token1        8 non-null      object
 3   fee           8 non-null      int64 
 4   tickspacing   8 non-null      int64 
dtypes: int64(2), object(3)
memory usage: 452.0+ bytes


In [33]:
position_curated_data = position_curated_data_prefinal.merge(pool_info_curated_data,how='left',left_on=['pool_address'],right_on=['pool_address'])
position_curated_data.info()
position_curated_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   block_timestamp   8 non-null      object 
 1   blockHash         8 non-null      object 
 2   transactionHash   8 non-null      object 
 3   blockNumber       8 non-null      int64  
 4   transactionIndex  8 non-null      int64  
 5   logIndex          8 non-null      int64  
 6   pool_address      8 non-null      object 
 7   owner             8 non-null      object 
 8   sender            2 non-null      object 
 9   liquidity         8 non-null      int64  
 10  amount0           8 non-null      int64  
 11  amount1           8 non-null      int64  
 12  pool_event        8 non-null      object 
 13  tickLower         8 non-null      int64  
 14  tickUpper         8 non-null      int64  
 15  NFT_event         6 non-null      object 
 16  tokenId           6 non-null      float64
 17  t

,block_timestamp,blockHash,transactionHash,blockNumber,transactionIndex,logIndex,pool_address,owner,sender,liquidity,...,amount1,pool_event,tickLower,tickUpper,NFT_event,tokenId,token0,token1,fee,tickspacing
0,2024-09-12T08:46:47.000Z,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,20733432,121,196,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,434507173001454,...,2498368860914347040,Burn,193380,200280,DecreaseLiquidity,807812.0,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,60
1,2024-09-12T08:46:47.000Z,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,20733432,121,198,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,0,...,0,Burn,193380,200280,NaN,NaN,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,60
2,2024-09-12T08:59:23.000Z,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,20733495,146,418,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,50402832039922,...,289810787703649019,Burn,193380,200340,DecreaseLiquidity,807811.0,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,60
3,2024-09-10T19:32:47.000Z,0x5643e2956fa116379a793bc3156e9980a932c309e65b...,0x44f9bb7c1cc5057a0888329e5399cef339182c2d6806...,20722319,7,96,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,100805664079844,...,579999999590987635,Mint,193380,200340,IncreaseLiquidity,807811.0,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,60
4,2024-09-25T23:51:23.000Z,0x161ed2d72668674a98cdd77da68842ecb83af2f6e53b...,0x38651d8b15f6b54645e486debf3a2a4908137d1bfe5b...,20831017,94,253,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,NaN,50402832039922,...,235673801212928920,Burn,193380,200340,DecreaseLiquidity,807811.0,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,60


In [34]:
position_curated_data.dropna(subset="tokenId",inplace=True)
position_curated_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 0 to 7
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   block_timestamp   6 non-null      object 
 1   blockHash         6 non-null      object 
 2   transactionHash   6 non-null      object 
 3   blockNumber       6 non-null      int64  
 4   transactionIndex  6 non-null      int64  
 5   logIndex          6 non-null      int64  
 6   pool_address      6 non-null      object 
 7   owner             6 non-null      object 
 8   sender            2 non-null      object 
 9   liquidity         6 non-null      int64  
 10  amount0           6 non-null      int64  
 11  amount1           6 non-null      int64  
 12  pool_event        6 non-null      object 
 13  tickLower         6 non-null      int64  
 14  tickUpper         6 non-null      int64  
 15  NFT_event         6 non-null      object 
 16  tokenId           6 non-null      float64
 17  token0

## Load Data

### BigQuery

In [35]:
from google.cloud import bigquery
from google.oauth2 import service_account

In [38]:
def load_to_bg(df:pd.DataFrame,credentials_file:str,table_id:str) -> None:
    try:
        credentials = service_account.Credentials.from_service_account_file(credentials_file)
        client = bigquery.Client(credentials=credentials, project=credentials.project_id)
        job = client.load_table_from_dataframe(df, table_id)
        logging.debug(f'Load to BG job result:{job.result()}')
        logging.info(f'data loaded to BG sucessfully!')
        return None
    except Exception as e:
        logging.error(f"BG error: {e}")


In [40]:
table_id = "db1-sendbox.uniswapV3.position_data"
credentials_file = r"BG_credentials.json"
bg_job_result = load_to_bg(position_curated_data,credentials_file,table_id)

2024-10-19 17:13:24 - INFO - data loaded to BG sucessfully!


### Postgres

In [41]:
postgres_password = KEYS['postgres_key']

In [42]:
def load_jsonfile(config_path:str)-> Any:
    with open(config_path, "r") as file:
        return json.load(file)

In [45]:
postgres_config_filepath= r'configs\PostgeSQL.json'
postgres_config = load_jsonfile(postgres_config_filepath)
host = postgres_config["host"]
dbname = postgres_config["dbname"]
user = postgres_config["user"]
port = postgres_config["port"]
table = postgres_config["tables"]["position_data_table_name"]

In [62]:
def initiate_connection_postgres(dbname:str,password:str,user:str,host:str,port:str) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | None:
    try:
        connection = psycopg2.connect(
        dbname= dbname ,
        user= user ,
        password= password ,
        host= host ,
        port= port 
                        )
        cursor = connection.cursor()
        logging.info("Postgres connection initiated")
        return connection,cursor
    
    except Exception as e:
        logging.error(f"initiate_connection_postgres: Error: {e}")
        return None

In [47]:
def enforce_data_type(data:pd.DataFrame, columns_and_types: list[tuple[list[str], str]],time_column:str) -> pd.DataFrame:
    df = data.copy()
    for columns, dtype in columns_and_types:
        for column in columns:
            if column in df.columns:
                df[column] = df[column].astype(dtype)
    if time_column:
        df[time_column] = pd.to_datetime(df[time_column], utc=True)
    return df

In [49]:
def create_postgrestable(connection, cursor ,query):
    try:
        cursor.execute(query)
        connection.commit()
        print("create_postgrestable: Table created successfully.")
    except Exception as e:
        print(f" create_postgrestable: Error: {e}")
        return None

In [50]:
def load_to_postgres(df, table_name,connection,cursor):
    try:
        buffer = StringIO()
        df.to_csv(buffer, index=False, header=False, sep='\t')
        buffer.seek(0)
        cursor.copy_from(buffer, table_name, sep='\t', null="NULL", columns=list(df.columns))
        connection.commit()
        print("Postgres: Data inserted successfully!")
    except Exception as e:
        connection.rollback()
        print(f"Error: {e}")
    return None

In [57]:
def lowercase_columns_df(data:pd.DataFrame) -> pd.DataFrame:
    df = data.copy()
    df.columns = df.columns.str.lower()
    return df

In [63]:
def prep_data_forpg(data:pd.DataFrame,columns_dtypes:list,time_column:str):
    df = enforce_data_type(data,columns_dtypes,time_column)
    df = lowercase_columns_df(df)
    return df

In [51]:
positions_data_table_creation_query = f"""
CREATE TABLE IF NOT EXISTS {table} (
    PRIMARY KEY (blockNumber, transactionIndex, logIndex),
    block_timestamp TIMESTAMP,
    blockHash VARCHAR,
    transactionHash VARCHAR,
    blockNumber BIGINT,
    transactionIndex INT,
    logIndex INT,
    pool_address VARCHAR,
    owner VARCHAR,
    sender VARCHAR,
    liquidity BIGINT,
    amount0 BIGINT,
    amount1 BIGINT,
    pool_event VARCHAR,
    tickLower INT,
    tickUpper INT,
    NFT_event VARCHAR,
    tokenid FLOAT,
    token0 VARCHAR,
    token1 VARCHAR,
    fee INT,
    tickspacing INT
    
);
"""
print(positions_data_table_creation_query)


CREATE TABLE IF NOT EXISTS position_data (
    PRIMARY KEY (blockNumber, transactionIndex, logIndex),
    block_timestamp TIMESTAMP,
    blockHash VARCHAR,
    transactionHash VARCHAR,
    blockNumber BIGINT,
    transactionIndex INT,
    logIndex INT,
    pool_address VARCHAR,
    owner VARCHAR,
    sender VARCHAR,
    liquidity BIGINT,
    amount0 BIGINT,
    amount1 BIGINT,
    pool_event VARCHAR,
    tickLower INT,
    tickUpper INT,
    NFT_event VARCHAR,
    tokenid FLOAT,
    token0 VARCHAR,
    token1 VARCHAR,
    fee INT,
    tickspacing INT
    
);



In [52]:
position_curated_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 0 to 7
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   block_timestamp   6 non-null      object 
 1   blockHash         6 non-null      object 
 2   transactionHash   6 non-null      object 
 3   blockNumber       6 non-null      int64  
 4   transactionIndex  6 non-null      int64  
 5   logIndex          6 non-null      int64  
 6   pool_address      6 non-null      object 
 7   owner             6 non-null      object 
 8   sender            2 non-null      object 
 9   liquidity         6 non-null      int64  
 10  amount0           6 non-null      int64  
 11  amount1           6 non-null      int64  
 12  pool_event        6 non-null      object 
 13  tickLower         6 non-null      int64  
 14  tickUpper         6 non-null      int64  
 15  NFT_event         6 non-null      object 
 16  tokenId           6 non-null      float64
 17  token0

In [65]:
time_column = "block_timestamp"
columns_dtypes = [(['blockhash', 'transactionhash', 'pool_address','owner','sender','pool_event','nft_event','token0','token1'], 'string'), 
                    (['liquidity', 'amount0', 'amount1','ticklower','tickupper','fee','tickspacing','transactionindex','logindex','tokenId'], 'int')]
position_curated_data_pg = prep_data_forpg(position_curated_data,columns_dtypes,time_column)
position_curated_data_pg.info()
position_curated_data_pg.head()
    

<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 0 to 7
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   block_timestamp   6 non-null      datetime64[ns, UTC]
 1   blockhash         6 non-null      object             
 2   transactionhash   6 non-null      object             
 3   blocknumber       6 non-null      int64              
 4   transactionindex  6 non-null      int64              
 5   logindex          6 non-null      int64              
 6   pool_address      6 non-null      string             
 7   owner             6 non-null      string             
 8   sender            2 non-null      string             
 9   liquidity         6 non-null      int64              
 10  amount0           6 non-null      int64              
 11  amount1           6 non-null      int64              
 12  pool_event        6 non-null      string             
 13  ticklower     

,block_timestamp,blockhash,transactionhash,blocknumber,transactionindex,logindex,pool_address,owner,sender,liquidity,...,amount1,pool_event,ticklower,tickupper,nft_event,tokenid,token0,token1,fee,tickspacing
0,2024-09-12 08:46:47+00:00,0x164db9b8a7c4c30294525f89e1e07d7919575cb5fde6...,0xa5315ea4b150820518ef430bec53646264b29cd996e8...,20733432,121,196,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,<NA>,434507173001454,...,2498368860914347040,Burn,193380,200280,DecreaseLiquidity,807812,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,60
2,2024-09-12 08:59:23+00:00,0x012c7620eb52bb9dcca3e7bc9540ee0fb697a6ba48d3...,0x3f60f11e8f1c05deab82badea8323e3d8bc72934c66e...,20733495,146,418,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,<NA>,50402832039922,...,289810787703649019,Burn,193380,200340,DecreaseLiquidity,807811,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,60
3,2024-09-10 19:32:47+00:00,0x5643e2956fa116379a793bc3156e9980a932c309e65b...,0x44f9bb7c1cc5057a0888329e5399cef339182c2d6806...,20722319,7,96,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,100805664079844,...,579999999590987635,Mint,193380,200340,IncreaseLiquidity,807811,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,60
4,2024-09-25 23:51:23+00:00,0x161ed2d72668674a98cdd77da68842ecb83af2f6e53b...,0x38651d8b15f6b54645e486debf3a2a4908137d1bfe5b...,20831017,94,253,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,<NA>,50402832039922,...,235673801212928920,Burn,193380,200340,DecreaseLiquidity,807811,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,60
5,2024-09-24 10:57:59+00:00,0x9ff02be3eb793f000e68148ba750e51485a9b058e68d...,0xbcce8ffd844ea8a0cf240fb373ae03a753359ac9621e...,20820005,21,186,0x6637d3d16bf761289e0e2dc7668135c49e9dd801,0xC36442b4a4522E871399CD717aBDD847Ab11FE88,<NA>,434507173001455,...,2009536441754882992,Burn,193380,200280,DecreaseLiquidity,807812,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,3000,60


In [66]:
connextion, cursor = initiate_connection_postgres(dbname,postgres_password,user,host,port)
create_postgrestable(connextion, cursor ,positions_data_table_creation_query)
load_to_postgres(lowercase_columns_df(enforce_data_type(position_curated_data_pg,columns_dtypes,'block_timestamp')), table ,connextion,cursor)

2024-10-19 18:08:55 - INFO - Postgres connection initiated


create_postgrestable: Table created successfully.
Error: ERREUR:  la valeur d'une clé dupliquée rompt la contrainte unique « position_data_pkey »
DETAIL:  La clé « (blocknumber, transactionindex, logindex)=(20733432, 121, 196) » existe déjà.
CONTEXT:  COPY position_data, ligne 1

